In [ ]:
import os, pickle, gc, warnings, copy, shutil, json, sys, re
from glob import glob
from itertools import product
warnings.filterwarnings('ignore')

import numpy as np 
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import StratifiedKFold
from tensorflow.config import list_physical_devices

from sklearn.metrics import log_loss

# Version log
## V0
- FS general v0
    - encoded
    - sscaled
    - mean impute
- Param idx = 1
- N-fold = 10
- Fit on
    - Original data
    - Extra data
    - FS with iv woe

In [ ]:
ORI_DIR = '/kaggle/input/icr-identify-age-related-conditions'
FE_DIR = '/kaggle/input/icr-fe-v3'
FS_DIR = '/kaggle/input/icr-fs-d3-iv-woe-v1'
IV_DIR = '/kaggle/input/icr-iv-woe-v2'
UTILS_DIR = '/kaggle/input/icr-utils'
sys.path.append(FE_DIR)
sys.path.append(IV_DIR)
sys.path.append(UTILS_DIR)

from iv_woe_feature_selection import get_ft_woe_mappings, woe_map_fts
from utils import load_pickle, save_pickle, get_model, train_model \
    , save_model, get_besttree, pred_multi_to_single, get_model_params \
    , get_all_class_weights, add_multiclass, add_greeks_cv_columns \
    , get_cv_col

In [ ]:
MODEL_TYPE = 'LR'
N_FOLD = 5
PARAM_IDX = 1
WOE_MAPPING = True

FS_RESULT = ['original']
FS_RESULT += sorted([
    url for corr_th in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    for url in glob(f'{FS_DIR}/corr{corr_th}/*.csv')
    if 'sulov' not in url
])
if FS_RESULT == 'all':
    FS_RESULT = sorted(glob(f'{FS_DIR}/*.csv'))
DEFAULT_FE_CONFIG = {
    'integerized': True
    , 'encoded': True
    , 'sscaled': True
    , 'imputing_strategy': None#'median' # mean, median, constant
    , 'epsilon_cols': [] # Epsilon cols to include
    , 'x_mul_y': True
    , 'x_div_y': True
    , 'drop_cols': []
}

In [ ]:
HAVING_FS = ('FS_DIR' in locals()) or ('FS_DIR' in globals())
if HAVING_FS:
    fe_path = os.path.join(FS_DIR, 'fe_config.pkl')
    if os.path.exists(fe_path):
        FE_config = load_pickle(fe_path)
    else:
        FE_config = DEFAULT_FE_CONFIG.copy()
else:
    FE_config = DEFAULT_FE_CONFIG.copy()
print(FE_config)
SEARCH_CONFIG = {
    'imbal_treater': ['oversampling', 'weighting']
    , 'cv': ['target', 'target_year', 'target_experiment']
    , 'multi_class': [True, False]
}
SEARCH_MODEL_CONFIG = {
    'RF': {
        'criterion': ['gini', 'entropy', 'log_loss']
    }
    , 'SVC': {
        'kernel': ['linear', 'poly', 'rbf', 'sigmoid']
    }
    , 'LR': {
        'multi_class': ['multinomial', 'ovr']
        , 'max_iter': [2000]
    }
}

RS = 719
USE_GPU = (len(list_physical_devices('GPU')) > 0)
if USE_GPU:
    print('Using GPU for training')
else:
    print('Using CPU')

In [ ]:
df_train = pd.read_csv(os.path.join(ORI_DIR, 'train.csv'))
df_test = pd.read_csv(os.path.join(ORI_DIR, 'test.csv'))
greeks = pd.read_csv(os.path.join(ORI_DIR, 'greeks.csv'))

# Feature Engineering

In [ ]:
# Feature Engineering
from feature_engineering import FeatureEngineer, check_original_df, check_transformed_df

save_pickle(FE_config, 'fe_config.pkl')
FE = FeatureEngineer(**FE_config)
    
df_train_fe = FE.fit_transform(df_train, is_training=True)
df_test_fe = FE.transform(df_test, is_training=False)

In [ ]:
check_original_df(df_train, FE)
df_train.head()

In [ ]:
check_transformed_df(df_train_fe, FE)
df_train_fe.head()

In [ ]:
check_transformed_df(df_test_fe, FE)
df_test_fe.head()

# Treating Class Imbalance

In [ ]:
df_train_processed = add_multiclass(df_train_fe)
df_train_processed = get_all_class_weights(df_train_processed)
df_train_processed[['Class', 'Multiclass', 'Class_weight', 'Multiclass_weight']].drop_duplicates()

# Stratified KFold

In [ ]:
df_train_processed = add_greeks_cv_columns(df_train_processed)
print(df_train_processed.year.value_counts())
print('='*30)
print(df_train_processed.experiment.value_counts())
df_train_processed.head()

# Grid Search

In [ ]:
from itertools import product
def my_product(inp):
    return list(dict(zip(inp.keys(), values)) for values in product(*inp.values()))

In [ ]:
FEATURES = {
    'binary': {}
    , 'multiclass': {}
}

def is_iv(fi_type):
    return ((fi_type[:3] == 'iv_') or (fi_type[:6] == 'ivwoe_'))

def get_features(name, path):
    fi = pd.read_csv(path)
    if ('tpi' in name) and ('_full' not in name):
        features = fi.loc[fi.importance > 0].feature.tolist()
    else:
        features = fi.feature.tolist()
    return features

for path in FS_RESULT:
    if path in ('original', 'all'):
        if path == 'original':
            fts = FE.num_cols + FE.cat_cols            
        else:
            fts = FE.features.copy()
        FEATURES['multiclass'][path] = fts
        FEATURES['binary'][path] = fts
    else: 
        basename = os.path.basename(path)
        corr = os.path.basename(os.path.dirname(path))
        tgt_type = re.search(r'_([a-z]+).csv', basename)[1]
        fi_type = corr + '_' + '_'.join(basename.split('_')[:-1])
        FEATURES[tgt_type][fi_type] = get_features(fi_type, path)
        if is_iv(fi_type):
            if 'tpi' in fi_type:
                fi_type += '_full'
                FEATURES[tgt_type][fi_type] = get_features(fi_type, path)

save_pickle(FEATURES, 'feature_sets')
                
def balanced_log_loss(y_true, y_pred):
    """ Calculate balance log loss """
    nc = np.bincount(y_true)
    return log_loss(y_true, y_pred, sample_weight = 1/nc[y_true], eps=1e-15)

def reset_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.mkdir(path)
    
SEARCH_PARAMS = SEARCH_CONFIG.copy()
SEARCH_PARAMS['feature_types'] = list(FEATURES[list(FEATURES.keys())[-1]].keys())
mdl_srch_cfg = SEARCH_MODEL_CONFIG.get(MODEL_TYPE, None)
if mdl_srch_cfg:
    SEARCH_PARAMS['model_param'] = my_product(mdl_srch_cfg.copy())
SRCH_SPACE = my_product(SEARCH_PARAMS)

print(f'Search space len: {len(SRCH_SPACE)}')
print('Search params:')
print(json.dumps(SEARCH_PARAMS, indent=4))

print()
print('Model param example')
print(get_model_params(
    MODEL_TYPE
    , param_idx=PARAM_IDX
    , multi_class=True
    , use_gpu=USE_GPU
    , rs=RS       
))
print()

In [ ]:
search_result = pd.DataFrame(
    columns=[
        'params', 'n_features', 'min_loss', 'max_loss'
        , 'std_loss', 'mean_loss', 'oof_loss'
    ]
)

y_true = df_train_processed.loc[:, 'Class']
y_true = (y_true > 0).astype(int)

oof = np.zeros((len(df_train_processed), 2))
grand_parent_dir = MODEL_TYPE
reset_dir(grand_parent_dir)

for s_idx, srch_params in enumerate(SRCH_SPACE):
    _ = gc.collect()
    print('#'*20)
    print(f'Search {s_idx}: {srch_params}')
    
    # Get parameters
    multiclass = srch_params['multi_class']
    cv_type = srch_params['cv']
    imb_treater = srch_params['imbal_treater']
    model_param = srch_params.get('model_param', {})
    ft_type = srch_params['feature_types']
    
    # Create variables
    target_col_name = 'Multiclass' if multiclass else 'Class'
    tgt_type = 'multiclass' if multiclass else 'binary'
    split_col = get_cv_col(df_train_processed, target_col_name, cv_type)
    features = FEATURES[tgt_type][ft_type]
    if imb_treater == 'weighting':
        weight_col_name = target_col_name + '_weight'
    
    # Create parent dir
    parent_dir = os.path.join(grand_parent_dir, f'search_{s_idx}')
    reset_dir(parent_dir)
    
    oof_search = oof.copy()
    losses = []
    
    # WoE mapping
    if WOE_MAPPING and ('corr' in ft_type):
        corr_lvl = float(re.search(r'corr(.+)_', ft_type)[1])

        woe = load_pickle(f'{IV_DIR}/WoE_{target_col_name}.pkl')
        woe_mappings = get_ft_woe_mappings(woe, features)
        df_train_final = woe_map_fts(df_train_processed, woe_mappings, features)
    else:
        df_train_final = df_train_processed.copy()
    
    # Break if not enough features
    n_features = len(features)
    print(f'N-features: {n_features}')
    if n_features == 0:
        print('Not enough features')
        continue
        
    skf = StratifiedKFold(n_splits=N_FOLD, shuffle=True, random_state=RS)
    splits = skf.split(df_train_final, split_col)
    params = get_model_params(
        MODEL_TYPE
        , param_idx=PARAM_IDX
        , multi_class=multiclass
        , use_gpu=USE_GPU
        , rs=RS
    )
    params.update(model_param)

    for idx, (train_index,val_index) in enumerate(splits):
        
        print(f'{idx} ', end='')
        
        # Get data
        X_train = df_train_final.loc[train_index, features]
        X_val = df_train_final.loc[val_index, features]
        y_train = df_train_final.loc[train_index, target_col_name]
        y_val = df_train_final.loc[val_index, target_col_name]
        
        if imb_treater == 'oversampling':
            ros = RandomOverSampler(random_state=RS)
            X_train, y_train = ros.fit_resample(X_train, y_train)
            weight_train = [1]*len(X_train)
            weight_val = [1]*len(X_val)
        elif imb_treater == 'weighting':
            weight_train = df_train_final.loc[train_index, weight_col_name].copy()
            weight_val = df_train_final.loc[val_index, weight_col_name].copy()
        
        # Get model
        model = get_model(MODEL_TYPE, params)
        
        # Train model
        model = train_model(
            MODEL_TYPE, model
            , X_train, y_train, weight_train
            , X_val, y_val, weight_val
            , params
        )
        
        # Save model
        best_ntree = get_besttree(MODEL_TYPE, model)
        
        # Inference
        preds = model.predict_proba(X_val)
        preds = pred_multi_to_single(preds, multi_class=multiclass)
        y_val = (y_val>0).astype(int)
        oof_search[val_index, :] += preds
        
        loss_fold = balanced_log_loss(y_val, preds)
        losses.append(loss_fold)
        print(f'({best_ntree})', end=', ')
    
    oof_loss = balanced_log_loss(y_true, oof_search)
    search_result.loc[len(search_result)] = [
        srch_params, n_features
        , np.min(losses), np.max(losses)
        , np.std(losses), np.mean(losses)
        , oof_loss
    ]
    print()
    print(f'OOF loss: {round(oof_loss,5)} (mean={round(np.mean(losses),5)}, std={round(np.std(losses),5)})')
    print()
search_result.to_csv('search_result.csv', index=False)

# Result

In [ ]:
print('Top 5 with <= 100 features')
tmp = search_result.loc[(search_result.n_features <= 100)].sort_values(by='mean_loss', ascending=True)
for idx in range(5):
    row = tmp.iloc[idx]
    print('#'*20)
    print(f'Search {row.name}')
    print('\t', row.params)
    print('\t', row.n_features, '\t', row.oof_loss)

print()
print('Top 5 with < 100 features')
tmp = search_result.loc[(search_result.n_features < 100)].sort_values(by='mean_loss', ascending=True)
for idx in range(5):
    row = tmp.iloc[idx]
    print('#'*20)
    print(f'Search {row.name}')
    print('\t', row.params)
    print('\t', row.n_features, '\t', row.oof_loss)

In [ ]:
for sort_col in ['mean_loss', 'oof_loss']:
    print('='*30)
    print(f'Overall top 5 param set by {sort_col}')
    tmp = search_result.sort_values(by=sort_col, ascending=True).reset_index(drop=True)
    for r in range(5):
        row = tmp.iloc[r]
        print(row['n_features'], '\t', round(row[sort_col], 5), '\t', row['params'])
    print()

In [ ]:
for cv in SEARCH_PARAMS['cv']:
    print('#'*20)
    print(f'Top by {cv}-CV')
    for sort_col in ['mean_loss', 'oof_loss']:
        print('='*30)
        print(f'Top 5 param set by {sort_col}')
        tmp = search_result.loc[search_result.params.apply(lambda x: x['cv'] == cv)]
        tmp = tmp.sort_values(by=sort_col, ascending=True).reset_index(drop=True)
        for r in range(5):
            row = tmp.iloc[r]
            print(row['n_features'], '\t', round(row[sort_col], 5), '\t', row['params'])
        print()